# Random Embedding + Supervised Head Training Baseline (STP1710.7 vs control)

This notebook runs the **random window embedding baseline** for the supervised-head training curve.

It does **not** use learned embedding values. It only needs file-level labels (`y_file`) and frog IDs (`frog_ids`).

Default target shape:

```python
N_FILES = 45
N_WINDOWS = 32
EMBED_DIM = 64
```

Important: if your loaded `y_file` has a different length, the notebook will automatically use `len(y_file)` for `N_FILES` to avoid shape mismatch. For STP1710.7 vs control, the binary subset may be fewer than all 43 usable LC--MS files.


In [ ]:
# ============================================================
# 1. Imports
# ============================================================
import os
import re
import random
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    balanced_accuracy_score,
)

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

# ============================================================
# 2. Configuration
# ============================================================

def seed_everything_all(seed: int = 0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# You said these should be correct.
# The notebook will auto-adjust N_FILES to len(y_file) after labels are loaded if needed.
N_FILES = 45
N_WINDOWS = 32
EMBED_DIM = 64

MAX_EPOCHS = 200
EVAL_EPOCHS = [10,20,30,40,50,60,70,80,90,100,110,120,130,140,150,160,170,180,190,200]

LR = 1e-3
WEIGHT_DECAY = 1e-4
RANDOM_SEEDS = range(20)

POS_LABEL = "STP1710.7"
NEG_LABEL = "control"

# Current server default path. Change only BASE_DIR if needed.

PROJECT_ROOT = Path("../..").resolve()
META_CSV =PROJECT_ROOT / "data" / "processed" / "metadata_with_frog.csv" 
NPZ_DIR=PROJECT_ROOT / "data" / "processed" /"mzML_npz_45"
OUT_DIR = PROJECT_ROOT /"results"/"runs"/"Supervised"/"Task1710"/"random_embedding"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 0
seed_everything_all(SEED)


print("META_CSV:", META_CSV, "exists=", META_CSV.exists())
print("NPZ_DIR:", NPZ_DIR, "exists=", NPZ_DIR.exists())
print("OUT_DIR:", OUT_DIR)



# ============================================================
# 3. Helpers for labels and frog IDs
# ============================================================

def normalize_uhm_from_dataset_filename(fn: str) -> str:
    """UHM18-10842_MS1.npz -> UHM18-10842"""
    s = os.path.basename(str(fn))
    s = re.sub(r"\.(npz|mzML)$", "", s)
    s = re.sub(r"_MS1$", "", s)
    return s


def frog_from_uhm_sample(uhm_sample: str) -> str:
    """UHM18-10842 -> UHM18"""
    return str(uhm_sample).split("-")[0]


def build_binary_file_labels(meta_csv, npz_dir, pos_label="STP1710.7", neg_label="control"):
    df = pd.read_csv(meta_csv, dtype=str)
    df["UHM_sample"] = df["UHM_sample"].astype(str).str.strip()
    df["treatment"] = df["treatment"].astype(str).str.strip()

    df = df[df["treatment"].isin([pos_label, neg_label])].copy()
    df["y"] = (df["treatment"] == pos_label).astype(int)

    uhm2label = dict(zip(df["UHM_sample"], df["y"]))
    uhm2treat = dict(zip(df["UHM_sample"], df["treatment"]))
    uhm2frog = {u: frog_from_uhm_sample(u) for u in df["UHM_sample"]}

    npz_files_all = sorted(Path(npz_dir).rglob("*.npz"))
    use_files = []
    for p in npz_files_all:
        uhm = normalize_uhm_from_dataset_filename(p.name)
        if uhm in uhm2label:
            use_files.append(p)

    rows = []
    y_list = []
    frog_list = []

    for p in use_files:
        uhm = normalize_uhm_from_dataset_filename(p.name)
        y = int(uhm2label[uhm])
        frog = uhm2frog[uhm]
        rows.append({
            "file_name": p.name,
            "path": str(p),
            "uhm_sample": uhm,
            "treatment": uhm2treat[uhm],
            "y": y,
            "frog_id": frog,
        })
        y_list.append(y)
        frog_list.append(frog)

    file_metadata = pd.DataFrame(rows)
    y_file = np.asarray(y_list, dtype=np.int64)
    frog_ids = np.asarray(frog_list)

    print("metadata label counts:")
    display(df["treatment"].value_counts())
    print("n npz all:", len(npz_files_all))
    print("n matched binary files:", len(use_files))
    print("n frogs:", len(np.unique(frog_ids)) if len(frog_ids) else 0)

    if len(use_files) == 0:
        print("WARNING: no NPZ files matched metadata. Check NPZ_DIR and file naming.")
        print("First metadata UHM_sample values:", list(df["UHM_sample"].head(10)))
        print("First NPZ normalized IDs:", [normalize_uhm_from_dataset_filename(p.name) for p in npz_files_all[:10]])

    return y_file, frog_ids, file_metadata



# ============================================================
# 4. Load y_file and frog_ids
# ============================================================
# Build labels directly from metadata + NPZ filenames.
# This avoids accidentally reusing old y_file.npy / frog_ids.npy
# from another task such as STP1710.7.

print("Building labels from metadata + NPZ filenames.")

y_file, frog_ids, file_metadata = build_binary_file_labels(
    META_CSV,
    NPZ_DIR,
    pos_label=POS_LABEL,
    neg_label=NEG_LABEL,
)

y_file = np.asarray(y_file, dtype=np.int64)
frog_ids = np.asarray(frog_ids)

print("y_file shape:", y_file.shape)
print("frog_ids shape:", frog_ids.shape)
print("label counts [0,1]:", np.bincount(y_file) if len(y_file) else None)
print("n frogs:", len(np.unique(frog_ids)))

assert len(y_file) > 0, "No labels loaded. Check META_CSV, NPZ_DIR, POS_LABEL, NEG_LABEL."
assert len(y_file) == len(frog_ids), "y_file and frog_ids length mismatch."

# Important: for binary STP1717.1 vs control, N_FILES may be less than 43.
if N_FILES != len(y_file):
    print(f"WARNING: N_FILES={N_FILES}, but len(y_file)={len(y_file)}. Using len(y_file).")
    N_FILES = len(y_file)

print("Final random embedding shape will be:", (N_FILES, N_WINDOWS, EMBED_DIM))

# Save labels into this output folder for reproducibility.
np.save(OUT_DIR / "y_file.npy", y_file)
np.save(OUT_DIR / "frog_ids.npy", frog_ids)
file_metadata.to_csv(OUT_DIR / "file_metadata.csv", index=False)



# ============================================================
# 5. Supervised head model
# ============================================================
class MeanPoolingClassifier(nn.Module):
    """
    Lightweight supervised head on frozen window embeddings.

    Input:  x with shape (batch, n_windows, embed_dim)
    Output: logits with shape (batch, 2)
    """
    def __init__(self, embed_dim, num_classes=2):
        super().__init__()
        self.norm = nn.LayerNorm(embed_dim)
        self.fc = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        x = x.mean(dim=1)
        x = self.norm(x)
        return self.fc(x)



# ============================================================
# 6. Train/evaluate random embedding supervised head with LOFO
# ============================================================

def train_random_supervised_head_one_seed(seed):
    seed_everything_all(seed)
    rng = np.random.default_rng(seed)

    # Random frozen window embeddings.
    # No learned embedding values are used here.
    E_rand = rng.normal(
        loc=0.0,
        scale=1.0,
        size=(N_FILES, N_WINDOWS, EMBED_DIM),
    ).astype(np.float32)

    logo = LeaveOneGroupOut()
    rows = []

    for fold_id, (train_idx, test_idx) in enumerate(logo.split(E_rand, y_file, groups=frog_ids), start=1):
        test_frog_unique = np.unique(frog_ids[test_idx])
        assert len(test_frog_unique) == 1

        X_train = torch.tensor(E_rand[train_idx], dtype=torch.float32, device=DEVICE)
        y_train = torch.tensor(y_file[train_idx], dtype=torch.long, device=DEVICE)

        X_test = torch.tensor(E_rand[test_idx], dtype=torch.float32, device=DEVICE)
        y_test_np = y_file[test_idx]
        test_frogs = frog_ids[test_idx]

        model = MeanPoolingClassifier(embed_dim=EMBED_DIM).to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

        for epoch in range(1, MAX_EPOCHS + 1):
            model.train()
            optimizer.zero_grad()
            logits = model(X_train)
            loss = F.cross_entropy(logits, y_train)
            loss.backward()
            optimizer.step()

            if epoch in EVAL_EPOCHS:
                model.eval()
                with torch.no_grad():
                    logits_test = model(X_test)
                    prob_test_file = torch.softmax(logits_test, dim=1)[:, POS_CLASS_INDEX].detach().cpu().numpy()

                # Aggregate file probabilities to frog level.
                for fg in np.unique(test_frogs):
                    m = test_frogs == fg
                    rows.append({
                        "seed": seed,
                        "fold": fold_id,
                        "epoch": epoch,
                        "test_frog": fg,
                        "frog_true": int(y_test_np[m][0]),
                        "frog_prob": float(prob_test_file[m].mean()),
                        "loss": float(loss.detach().cpu().item()),
                        "n_test_files": int(m.sum()),
                    })

    return pd.DataFrame(rows)


def summarize_epoch_metrics(df_fold):
    rows = []

    for epoch, g in df_fold.groupby("epoch"):
        y_true = g["frog_true"].to_numpy()
        y_prob = g["frog_prob"].to_numpy()
        y_pred = (y_prob >= 0.5).astype(int)

        rows.append({
            "epoch": int(epoch),
            "AUROC": roc_auc_score(y_true, y_prob),
            "AUPRC": average_precision_score(y_true, y_prob),
            "ACC": accuracy_score(y_true, y_pred),
            "Balanced_ACC": balanced_accuracy_score(y_true, y_pred),
            "n_frogs": len(y_true),
            "n_pos_frogs": int(y_true.sum()),
            "n_neg_frogs": int((1 - y_true).sum()),
        })

    return pd.DataFrame(rows).sort_values("epoch")




# ============================================================
# 7. Run random embedding baseline across seeds
# ============================================================
all_fold_outputs = []
all_epoch_metrics = []
POS_CLASS_INDEX = 0

for seed in RANDOM_SEEDS:
    print(f"Running random embedding supervised-head baseline, seed={seed} ...")
    df_fold = train_random_supervised_head_one_seed(seed)
    df_epoch = summarize_epoch_metrics(df_fold)

    df_epoch["seed"] = seed
    df_epoch["embedding"] = "random_embedding"
    df_epoch["n_files"] = N_FILES
    df_epoch["n_windows"] = N_WINDOWS
    df_epoch["embed_dim"] = EMBED_DIM

    all_fold_outputs.append(df_fold)
    all_epoch_metrics.append(df_epoch)

df_random_fold = pd.concat(all_fold_outputs, ignore_index=True)
df_random_epoch = pd.concat(all_epoch_metrics, ignore_index=True)

fold_csv = OUT_DIR / "random_supervised_head_fold_outputs.csv"
epoch_csv = OUT_DIR / "random_supervised_head_epoch_metrics_per_seed.csv"

df_random_fold.to_csv(fold_csv, index=False)
df_random_epoch.to_csv(epoch_csv, index=False)

print("Saved fold outputs to:", fold_csv)
print("Saved epoch metrics to:", epoch_csv)

display(df_random_epoch)


# ============================================================
# 8. Curve summary: mean ± 95% CI across random seeds
# ============================================================

def summarize_curve_ci(df, metric_cols=("AUROC", "AUPRC", "ACC", "Balanced_ACC")):
    rows = []

    for epoch, g in df.groupby("epoch"):
        row = {
            "epoch": int(epoch),
            "n_runs": len(g),
        }

        for m in metric_cols:
            mean = g[m].mean()
            sd = g[m].std(ddof=1)
            se = sd / np.sqrt(len(g))
            ci95 = 1.96 * se

            row[f"{m}_mean"] = mean
            row[f"{m}_sd"] = sd
            row[f"{m}_se"] = se
            row[f"{m}_ci95"] = ci95
            row[f"{m}_lower"] = mean - ci95
            row[f"{m}_upper"] = mean + ci95

        rows.append(row)

    return pd.DataFrame(rows).sort_values("epoch")


df_random_curve = summarize_curve_ci(df_random_epoch)
curve_csv = OUT_DIR / "random_supervised_head_epoch_curve_summary.csv"
df_random_curve.to_csv(curve_csv, index=False)

print("Saved curve summary to:", curve_csv)
display(df_random_curve)




## Label loading

The notebook first tries to load saved labels from `y_file.npy` and `frog_ids.npy`. If these are not found, it builds labels from the metadata CSV and matching NPZ filenames.


In [ ]:
# ============================================================
# 9. Optional quick plot: random baseline AUROC curve
# ============================================================
import matplotlib.pyplot as plt

plt.figure(figsize=(5.5, 3.4))
plt.plot(df_random_curve["epoch"], df_random_curve["AUROC_mean"], marker="o", label="Random embedding baseline")
plt.fill_between(
    df_random_curve["epoch"].to_numpy(),
    df_random_curve["AUROC_lower"].to_numpy(),
    df_random_curve["AUROC_upper"].to_numpy(),
    alpha=0.15,
)
plt.xlabel("Supervised head training epoch")
plt.ylabel("AUROC")
plt.title("Random embedding + supervised head baseline")
plt.grid(True, alpha=0.25)
plt.legend(frameon=False)
plt.tight_layout()

fig_path = OUT_DIR / "random_supervised_head_AUROC_curve.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
print("Saved figure to:", fig_path)
plt.show()
